In [1]:
# lesson_01 > unit_03 > multi_turm.py

# from openai import OpenAI
# from dotenv import load_dotenv

# load_dotenv()
# client = OpenAI()

# def get_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         temperature=0.9,
#         messages=messages,
#     )
#     return response.choices[0].message.content

# messages = [ {"role": "system", "content": "너는 사용자를 도와주는 상담사야."} ]

# while True:
#     user_input = input("사용자: ")

#     if user_input == "exit":
#         break

#     messages.append({"role": "user", "content": user_input})
#     ai_response = get_response(messages)  # 대화 기록을 기반으로 AI 응답 가져오기
#     messages.append({"role": "assistant", "content": str(ai_response)})

#     print("AI: " + str(ai_response)) 

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv("../../.env")

model = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 금융 상담 역할
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        #SystemMessage("당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다.")
        ("placeholder", "{messages}"),  # 대화 이력 추가
        # MessagesPlaceholder("messages")
    ]
)

# 프롬프트와 모델을 연결하여 체인 생성
chain = prompt | model

# 이전 대화를 포함한 메시지 전달
ai_msg = chain.invoke(
    {
        "messages": [
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"),
            # HumanMessage("저축을 늘리기 위해 무엇을 할 수 있나요?"),
            ("assistant", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),  # 챗봇의 답변
            # AIMessage("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),
            ("human", "방금 뭐라고 했나요?"),  # 사용자의 재확인 질문
            # HumanMessage("방금 뭐라고 했나요?"),
        ],
    }
)
print(ai_msg.content)  # 챗봇의 응답 출력

저축을 늘리기 위해서는 저축 목표를 설정하고, 매달 자동 이체를 통해 일정 금액을 저축하는 방법을 추천드립니다. 이렇게 하면 꾸준히 저축을 할 수 있습니다. 또한 다른 방법도 고려해보실 수 있습니다:

1. **예산 작성**: 월간 지출을 계획하고 필요한 부분과 불필요한 지출을 구분하여 절약할 수 있는 항목을 찾으세요.
   
2. **지출 추적**: 매일 지출을 기록하여 어디에서 돈이 나가는지 파악하고, 필요한 경우 생활 속에서 작은 지출을 줄여보세요.

3. **소액 저축**: 매일 작은 금액을 저축하는 습관을 들이면, 나중에 큰 금액이 될 수 있습니다. 예를 들어, 매일 1,000원씩 저축하면 한 달에 30,000원이 됩니다.

4. **스마트폰 앱 사용**: 저축 목표를 설정하고, 지출을 관리할 수 있는 재무 관리 앱을 사용하여 보다 체계적으로 저축을 할 수 있습니다.

5. **신용카드 사용 줄이기**: 신용카드 사용을 줄이고, 현금이나 체크카드로 소비하여 과소비를 방지하세요.

6. **불필요한 구독 서비스 취소**: 사용하지 않는 구독 서비스나 자동 결제가 있는 항목을 점검하고 취소하여 비용을 절감하세요.

이런 방법들을 통해 저축을 늘려보세요!


#### InMemoryChatMessageHistory

In [2]:
# from langchain_community.chat_message_histories import ChatMessageHistory <== 과거버전
from langchain_core.chat_history import InMemoryChatMessageHistory

# 대화 이력 저장을 위한 클래스 초기화
chat_history = InMemoryChatMessageHistory()

# 사용자 메시지 추가
chat_history.add_user_message("저축을 늘리기 위해 무엇을 할 수 있나요?")
chat_history.add_ai_message("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요.")

# 새로운 질문 추가 후 다시 체인 실행
chat_history.add_user_message("방금 뭐라고 했나요?")
ai_response = chain.invoke({"messages": chat_history.messages})
print(ai_response.content)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23188\2462301689.py:5: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  chat_history = InMemoryChatMessageHistory()


저축을 늘리기 위해 저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하는 것을 추천했습니다. 이렇게 하면 체계적으로 저축을 늘릴 수 있습니다. 더 궁금한 점이 있으신가요?


#### RunnableWithMessageHistory를 사용한 자동 대화 이력 관리

In [3]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)

# 대화 이력을 관리할 체인 설정
chat_history = InMemoryChatMessageHistory()
chain = prompt | model

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# 질문 메시지 체인 실행
answer = chain_with_message_history.invoke(
    {"input": "저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable": {"session_id": "user01"}},
).content

print(answer)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23188\296379841.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
d:\source\ai\langchain_app\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


저축을 늘리기 위해 다음과 같은 몇 가지 방법을 고려할 수 있습니다.

1. **예산 수립**: 자신의 수입과 지출을 명확하게 파악하고, 월별 예산을 세워 지출을 관리하세요. 필요 없는 지출을 줄이는 것이 중요합니다.

2. **자동 이체 설정**: 매달 급여가 들어오면 일정 금액을 자동으로 저축 계좌로 이체하도록 설정하세요. 이렇게 하면 저축하는 것을 잊지 않을 수 있습니다.

3. **긴급 자금 마련**: 예상하지 못한 지출에 대비해 긴급 자금을 마련하세요. 이렇게 하면 원하지 않는 소비를 줄일 수 있습니다.

4. **소비 패턴 분석**: 자신의 소비 습관을 분석하고, 군더더기 없는 소비를 실천하세요. 예를 들어, 외식을 줄이거나 불필요한 구독 서비스를 취소하는 방식입니다.

5. **고정 지출 최소화**: 불필요한 고정 지출(예: 고가의 요금제, 보험 등)을 검토해보고, 더 저렴한 옵션으로 변경하는 것도 방법입니다.

6. **보너스 및 추가 수입 활용**: 연말 보너스나 추가 수입이 들어올 때, 이를 저축에 활용하는 것도 좋은 방법입니다.

7. **저축 목표 설정**: 단기 및 장기 저축 목표를 설정하면 저축에 대한 동기를 부여할 수 있습니다. 목표에 따라 다르게 저축할 수 있습니다.

8. **재무 교육**: 금융 관련 서적이나 강의, 온라인 자료를 통해 재무 지식을 늘리고, 더 나은 투자 및 저축 전략을 배울 수 있습니다.

이러한 방법들을 종합적으로 실천하면, 저축을 효과적으로 늘릴 수 있습니다.


In [4]:
answer = chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"}, 
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content
print(answer)

당신은 "저축을 늘리기 위해 무엇을 할 수 있나요?"라고 질문하셨습니다. 저축을 늘리기 위한 여러 가지 방법을 제안드렸습니다. 추가로 궁금한 점이 있으시면 말씀해 주세요!


In [5]:
answer = chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"}, 
    {"configurable": {"session_id": "user02"}}  # 세션 ID 설정
).content
print(answer)

당신은 "내가 방금 뭐라고 했나요?"라고 질문하셨습니다. 이전에는 저축을 늘리는 방법에 대해 물어보셨습니다. 추가적으로 궁금한 사항이 있으시면 언제든지 말씀해 주세요!
